In [1]:
import pandas as pd
import numpy as np
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [3]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [4]:
#cleaning
def clean(text):
    text = str(text).lower()

    for p in string.punctuation:
        text = text.replace(p, "")
        
    return text

In [5]:
#Cleaning of columns
cols = ["prompt", "A", "B", "C", "D", "E"]

for col in cols:
    train[col] = train[col].apply(clean)
    test[col] = test[col].apply(clean)

In [6]:
#building 
collection = []

In [7]:
for i in range(len(train)):
    text = (
        train.loc[i, "prompt"] + " " +
        train.loc[i, "A"] + " " +
        train.loc[i, "B"] + " " +
        train.loc[i, "C"] + " " +
        train.loc[i, "D"] + " " +
        train.loc[i, "E"]
    )
    
    collection.append(text)

In [8]:
for i in range(len(test)):
    text = (
        test.loc[i, "prompt"] + " " +
        test.loc[i, "A"] + " " +
        test.loc[i, "B"] + " " +
        test.loc[i, "C"] + " " +
        test.loc[i, "D"] + " " +
        test.loc[i, "E"]
    )
    
    collection.append(text)

In [9]:
#model building
tfidf = TfidfVectorizer(stop_words="english")
tfidf.fit(collection)
print(len(tfidf.vocabulary_))

2865


In [10]:
def map_score(actual, preds):
    if actual in preds:
        rank = preds.index(actual) + 1
        return 1 / rank
    return 0

In [11]:
#score
labels = ["A", "B", "C", "D", "E"]
scores = []

In [12]:
for i in range(len(train)):
    prompt = train.loc[i, "prompt"]
    sim_scores = []
    
    for label in labels:
        option = train.loc[i, label]
        
        vectors = tfidf.transform([prompt, option])
        sim = cosine_similarity(vectors[0], vectors[1])[0][0]
        
        sim_scores.append((label, sim))
    
    sim_scores.sort(key=lambda x: x[1], reverse=True)
    
    top3 = [
        sim_scores[0][0],
        sim_scores[1][0],
        sim_scores[2][0]
    ]
    
    actual = train.loc[i, "answer"]
    scores.append(map_score(actual, top3))

In [13]:
print("Train MAP@3:", sum(scores)/len(scores))

Train MAP@3: 0.28225


In [14]:
predictions = []
labels = ["A", "B", "C", "D", "E"]

In [15]:
for i in range(len(test)):
    prompt = test.loc[i, "prompt"]
    sim_scores = []
    
    for label in labels:
        option = test.loc[i, label]
        
        vectors = tfidf.transform([prompt, option])
        sim = cosine_similarity(vectors[0], vectors[1])[0][0]
        
        sim_scores.append((label, sim))
    
    sim_scores.sort(key=lambda x: x[1], reverse=True)
    
    top3 = (
        sim_scores[0][0] + " " +
        sim_scores[1][0] + " " +
        sim_scores[2][0]
    )
    
    predictions.append(top3)

In [16]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": predictions
})

In [17]:
submission.to_csv("submission.csv", index=False)